In [1]:
# Imports
import json
import time
import urllib.request
from itertools import combinations
from pathlib import Path
from typing import Literal, NamedTuple

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pyomo.environ as pyo
from ortools.sat.python import cp_model
from pydantic import BaseModel, Field, model_validator
from pyomo.contrib.appsi.solvers.highs import Highs

---
# Part 0: Hello CP-SAT

CP-SAT is **declarative**: we describe *what* a solution looks like (variables, constraints, objective), and the solver figures out *how* to find it.

Every CP-SAT program follows the same four steps:
1. Create a `CpModel`.
2. Add variables: `new_bool_var`, `new_int_var`, ...
3. Add constraints and, optionally, an objective.
4. Solve with a `CpSolver` and read the values.

All variables are **integers** or **Booleans**. CP-SAT has no continuous variables. If you have fractional data, scale it, e.g. work in minutes instead of hours.

In [2]:
model = cp_model.CpModel()

x = model.new_int_var(0, 10, "x")
y = model.new_int_var(0, 10, "y")
b = model.new_bool_var("b")

model.add(x + y <= 12)
model.add(x >= 7).only_enforce_if(b)   # if b is true, then x >= 7
model.add(y >= 7).only_enforce_if(~b)  # if b is false, then y >= 7

model.maximize(2 * x + 3 * y)

solver = cp_model.CpSolver()
status = solver.solve(model)

print(f"Status: {solver.status_name(status)}")
print(f"x = {solver.value(x)}, y = {solver.value(y)}, b = {solver.value(b)}")
print(f"Objective: {solver.objective_value}")

Status: OPTIMAL
x = 2, y = 10, b = 0
Objective: 34.0


We have the following problem.

- There are N teams and K tasks.
- Every tasks has a release date, a duration and a due date. We choose the start time.
- Not every team can do every task, they need to be *compatible*.
- Not every team can work at every time, they have *unavailabilities*.
- Some tasks can only start if others have finished (*precendence constraints*).
- Some tasks need to be served by the same teams.

- Lexicographical optimization:
- First, minimize the number of teams in action.
- Second, minimize unfairness: Maximum working time - minimum working time among teams.

In [3]:
# Utility function for loading the challenge instances
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# !!    You can skip this Code      !!
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

DATA_URL = "https://raw.githubusercontent.com/jpjj/O4A-Hands-On-CP-SAT/main/data"


def load_instance(problem: str, name: str) -> dict:
    """Load an instance from the local data folder or, e.g. in Colab, from GitHub."""
    local = Path("data") / problem / f"{name}.json"
    if local.exists():
        return json.loads(local.read_text())
    with urllib.request.urlopen(f"{DATA_URL}/{problem}/{name}.json") as response:
        return json.loads(response.read())

In [4]:
data = load_instance("workforce", "instance_19")

In [6]:
data.keys()

dict_keys(['name', 'horizon', 'num_teams', 'tasks', 'same_team_groups', 'precedences', 'unavailabilities'])

# Define Problem

In [ ]:
from typing import NewType

TeamId = NewType("TeamId", int)
TaskId = NewType("TaskId", int)

class TimeIntervall(BaseModel):
    start: int
    end: int

class Team(BaseModel):
    id: TeamId
    unavailabilities: list[TimeIntervall]

class Task(BaseModel):
    id: TaskId
    release: int
    due: int
    duration: int
    compatible_teams: list[TeamId]


class WorkforceProblem(BaseModel):
    teams: list[Team]
    tasks: list[Task]
    precedence: list[tuple[TaskId, TaskId]] # first task needs to finish before second task can start
    same_team: list[list[TaskId]]

    @model_validator(mode="after")
    def check_feasible(self) -> "WorkforceProblem":
        # Check that precedence make sense given release and duration of the first and due and duration of the second
        id_to_task = {task.id: task for task in self.tasks}
        if overlapping_tasks:
            raise ValueError("Worker Shift contains overlapping tasks")
        if self.start > self.end:
            raise ValueError("Start time after end time for WorkerShift")
        if self.tasks:
            earliest_task_start = min(t.start for t in self.tasks)
            if earliest_task_start < self.start:
                raise ValueError("At least one task starts before WorkerShift")
            latest_task_end = max(t.end for t in self.tasks)
            if latest_task_end > self.end:
                raise ValueError("At least one task ends after WorkerShift")
        return self



class WorkforceSolution(BaseModel):
    assignment: dict[TaskId, TeamId]
    start_times: dict[TaskId, TeamId]

    
